# Bài 3 — Guardrails và hành động nguy hiểm

**Tuần 22** · Bài tập kèm theo: `ex03_guardrails.py`

Với chatbot, hậu quả tệ nhất là một câu trả lời sai.

Với agent, hậu quả tệ nhất là **một đơn hàng bị huỷ nhầm**, **một email gửi cho
sai người**, hoặc **10.000 món hàng được đặt vì model đọc nhầm một con số**.

> **Nguyên tắc trung tâm của cả bài:**
> **KHÔNG BAO GIỜ TIN THAM SỐ DO MODEL SINH RA.**
> Model viết `{"so_luong": 10000}` dễ dàng y hệt viết `{"so_luong": 10}`.

Không cần API key.

In [ ]:
import sys
from pathlib import Path

GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
P09 = GOC / "curriculum" / "09-agents"
sys.path.insert(0, str(P09))

from the_gioi import (CONG_CU_CHI_DOC, CONG_CU_KHONG_HOAN_TAC,
                      CONG_CU_THAY_DOI, TheGioi)

# Windows: console cp1252 lam sap moi lenh print tieng Viet
import sys as _s
for _l in (_s.stdout, _s.stderr):
    if (getattr(_l, "encoding", "") or "").lower().replace("-", "") != "utf8":
        try:
            _l.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

tg = TheGioi()
print(f"{len(tg.don_hang)} đơn hàng · {len(tg.kho)} mặt hàng")
print(f"Công cụ chỉ đọc: {sorted(CONG_CU_CHI_DOC)}")
print(f"Công cụ THAY ĐỔI thế giới: {sorted(CONG_CU_THAY_DOI)}")

In [ ]:
def dang_ky(the_gioi):
    return {
        "tra_cuu_don": the_gioi.tra_cuu_don,
        "tra_cuu_kho": the_gioi.tra_cuu_kho,
        "liet_ke_don": the_gioi.liet_ke_don,
        "cap_nhat_trang_thai": the_gioi.cap_nhat_trang_thai,
        "huy_don": the_gioi.huy_don,
        "dat_hang_bo_sung": the_gioi.dat_hang_bo_sung,
        "gui_email": the_gioi.gui_email,
    }


def thuc_thi(ten, tham_so, so_dang_ky):
    """Khong bao gio nem exception ra ngoai - model se go sai, do la binh thuong."""
    ham = so_dang_ky.get(ten)
    if ham is None:
        return {"ok": False,
                "loi": f"Không có công cụ {ten!r}. Có: {', '.join(sorted(so_dang_ky))}"}
    try:
        return ham(**(tham_so or {}))
    except TypeError as e:
        return {"ok": False, "loi": f"Sai tham so cho {ten}: {e}"}
    except Exception as e:
        return {"ok": False, "loi": f"{type(e).__name__}: {e}"}


def chay_agent(nhiem_vu, goi_model, so_dang_ky, max_vong=6):
    lich_su = [{"vai": "nhiem_vu", "noi_dung": nhiem_vu}]
    so_vong = 0
    for _ in range(max_vong):
        so_vong += 1
        try:
            qd = goi_model(lich_su)
        except Exception:
            return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
                    "ly_do_dung": "loi_model"}
        if not isinstance(qd, dict) or qd.get("loai") not in ("cong_cu", "tra_loi"):
            return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
                    "ly_do_dung": "loi_model"}
        if qd["loai"] == "tra_loi":
            return {"ket_qua": qd.get("noi_dung", ""), "duong_di": lich_su,
                    "so_vong": so_vong, "ly_do_dung": "tra_loi"}
        ten, ts = qd.get("ten", ""), qd.get("tham_so") or {}
        lich_su.append({"vai": "cong_cu", "ten": ten, "tham_so": ts})
        lich_su.append({"vai": "ket_qua", "noi_dung": thuc_thi(ten, ts, so_dang_ky)})
    return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
            "ly_do_dung": "het_vong"}


def in_duong_di(kq):
    n = 0
    ket_qua = [b["noi_dung"] for b in kq["duong_di"] if b["vai"] == "ket_qua"]
    for b in kq["duong_di"]:
        if b["vai"] != "cong_cu":
            continue
        ts = ", ".join(f"{k}={v!r}" for k, v in b["tham_so"].items())
        r = ket_qua[n] if n < len(ket_qua) else {}
        dau = "ok" if r.get("ok") else f'LOI: {r.get("loi", "?")}'
        n += 1
        print(f"{n}. {b['ten']}({ts}) -> {dau}")
    print(f"=> {kq['ket_qua'] or '(không có câu trả lời)'}   [{kq['ly_do_dung']}]")

## 1. Chuyện gì xảy ra khi không có guardrail

Hãy để một agent "hơi ngốc" chạy tự do trong thế giới thật. Đây là kiểu lỗi model
mắc thường xuyên, không phải trường hợp hiếm.

In [ ]:
tg = TheGioi()
sdk = dang_ky(tg)

def model_ngoc(lich_su):
    """Model hiểu 'đặt thêm cho đủ 50' thành 'đặt 5000'.

    Lỗi đọc nhầm đơn vị / nhầm cột là lỗi model mắc rất thường xuyên.
    """
    da_lam = [b["ten"] for b in lich_su if b["vai"] == "cong_cu"]
    if "tra_cuu_kho" not in da_lam:
        return {"loai": "cong_cu", "ten": "tra_cuu_kho",
                "tham_so": {"ma_hang": "VX-DEN-03"}}
    if "dat_hang_bo_sung" not in da_lam:
        return {"loai": "cong_cu", "ten": "dat_hang_bo_sung",
                "tham_so": {"ma_hang": "VX-DEN-03", "so_luong": 5000}}
    return {"loai": "tra_loi", "noi_dung": "Đã đặt thêm hàng."}

kq = chay_agent("Kho VX-DEN-03 hết rồi, đặt thêm cho đủ 50", model_ngoc, sdk)
in_duong_di(kq)
print()
print("Tồn kho sau khi agent chạy:", tg.kho["VX-DEN-03"]["ton"])

Ở đây thế giới của chúng ta **tự chặn** (giới hạn 100 mỗi lần), nên thiệt hại
không xảy ra. Nhưng hãy nhìn kỹ vào điều đó:

> ⚠️ **Bạn vừa được cứu bởi một `if` nằm bên trong công cụ.**
>
> Trong hệ thống thật, công cụ của bạn thường là một câu lệnh SQL, một lời gọi
> HTTP, hoặc một thao tác file — và nó sẽ **làm đúng những gì bạn đưa vào**. Nếu
> lớp bảo vệ duy nhất nằm rải rác bên trong từng công cụ, thì công cụ thứ tám bạn
> viết vội lúc 6 giờ chiều sẽ không có nó.

Guardrail phải là **một lớp riêng, nằm giữa model và mọi công cụ** — để không
phụ thuộc vào việc bạn có nhớ hay không.

## 2. Bốn cửa, đúng thứ tự

```
model đề xuất hành động
        │
        ▼  ① QUYỀN      công cụ này có được phép dùng không?
        ▼  ② THAM SỐ    giá trị model đưa có hợp lệ không?
        ▼  ③ NGÂN SÁCH  đã thay đổi thế giới quá nhiều lần chưa?
        ▼  ④ XÁC NHẬN   việc này có cần hỏi người không?
        ▼
    công cụ chạy thật
```

Thứ tự này **không tuỳ tiện**:

| Cặp | Vì sao theo thứ tự đó |
|---|---|
| Quyền **trước** tham số | Không tiết lộ định dạng tham số của công cụ mà người gọi vốn không được dùng |
| Ngân sách **trước** xác nhận | Hỏi người rồi mới báo hết ngân sách là làm phiền vô ích — và người dùng sẽ học cách bấm "có" cho nhanh, tức là bạn vừa phá huỷ chính lớp bảo vệ của mình |

In [ ]:
import re
from dataclasses import dataclass, field

MAU_MA_DON = re.compile(r"^DH\d{4}$")
MAU_MA_HANG = re.compile(r"^VX-[A-Z]+-\d{2}$")


def kiem_tra_tham_so(ten, tham_so):
    """Kiểm TRƯỚC khi chạy. Đây là lớp duy nhất đứng giữa model và hệ thống thật."""
    ts = tham_so or {}
    if ten in ("tra_cuu_don", "cap_nhat_trang_thai", "huy_don"):
        ma = ts.get("ma_don")
        if not isinstance(ma, str) or not MAU_MA_DON.match(ma):
            return {"ok": False, "loi": f"ma_don phải dạng DH+4 số, nhận được {ma!r}"}
    if ten in ("tra_cuu_kho", "dat_hang_bo_sung"):
        ma = ts.get("ma_hang")
        if not isinstance(ma, str) or not MAU_MA_HANG.match(ma):
            return {"ok": False, "loi": f"ma_hang phải dạng VX-CHU-SO, nhận được {ma!r}"}
    if ten == "dat_hang_bo_sung":
        sl = ts.get("so_luong")
        # isinstance(True, int) là True - quên loại bool thì True lọt qua thành 1
        if isinstance(sl, bool) or not isinstance(sl, int):
            return {"ok": False, "loi": f"so_luong phải là số nguyên, nhận được {sl!r}"}
        if not 1 <= sl <= 100:
            return {"ok": False, "loi": f"so_luong phải từ 1 đến 100, nhận được {sl}"}
    if ten == "gui_email":
        if "@" not in str(ts.get("dia_chi", "")):
            return {"ok": False, "loi": f"dia_chi không hợp lệ: {ts.get('dia_chi')!r}"}
        if not (ts.get("noi_dung") or "").strip():
            return {"ok": False, "loi": "noi_dung email không được rỗng"}
    return {"ok": True}


for ten, ts in [("huy_don", {"ma_don": "DH1002"}),
                ("huy_don", {"ma_don": "đơn của chị Bích"}),
                ("dat_hang_bo_sung", {"ma_hang": "VX-DEN-03", "so_luong": 50}),
                ("dat_hang_bo_sung", {"ma_hang": "VX-DEN-03", "so_luong": 5000}),
                ("dat_hang_bo_sung", {"ma_hang": "VX-DEN-03", "so_luong": True}),
                ("gui_email", {"dia_chi": "khach-hang", "noi_dung": "x"})]:
    r = kiem_tra_tham_so(ten, ts)
    print(f'{"✅" if r["ok"] else "❌"} {ten:<18} {str(ts)[:44]:<46} {r.get("loi", "")}')

> 📌 Thông báo lỗi **nhắc lại giá trị nhận được** là có chủ đích: người đọc nó
> là **model**, ở vòng lặp tiếp theo. *"so_luong phải từ 1 đến 100, nhận được
> 5000"* cho nó đủ thông tin để sửa ngay; *"Tham số không hợp lệ"* thì nó chỉ có
> thể đoán — và thường đoán sai rồi lặp lại đúng lỗi cũ.

## 3. Bộ kiểm soát

In [ ]:
CONG_CU_DOI_THE_GIOI = {"cap_nhat_trang_thai", "huy_don",
                        "dat_hang_bo_sung", "gui_email"}


@dataclass
class ChinhSach:
    cong_cu_cho_phep: set = None          # None = tất cả
    can_xac_nhan: set = field(default_factory=set)
    max_thay_doi: int = 3


class BoKiemSoat:
    def __init__(self, so_dang_ky, chinh_sach, ham_xac_nhan=None):
        self.so_dang_ky = so_dang_ky
        self.cs = chinh_sach
        self.ham_xac_nhan = ham_xac_nhan
        self.nhat_ky_chan = []
        self.da_thay_doi = []

    def _chan(self, ten, ly_do, thong_bao):
        self.nhat_ky_chan.append({"ten": ten, "ly_do": ly_do})
        return {"ok": False, "loi": thong_bao}

    def thuc_thi(self, ten, tham_so):
        ts = tham_so or {}

        if self.cs.cong_cu_cho_phep is not None and ten not in self.cs.cong_cu_cho_phep:
            return self._chan(ten, "quyen", f"Công cụ {ten!r} không được phép")

        kiem = kiem_tra_tham_so(ten, ts)
        if not kiem["ok"]:
            self.nhat_ky_chan.append({"ten": ten, "ly_do": "tham_so"})
            return kiem

        if ten in CONG_CU_DOI_THE_GIOI and len(self.da_thay_doi) >= self.cs.max_thay_doi:
            return self._chan(ten, "ngan_sach",
                              f"Hết ngân sách {self.cs.max_thay_doi} hành động")

        if ten in self.cs.can_xac_nhan:
            # None -> TỪ CHỐI. Mặc định phải an toàn: quên cấu hình thì agent
            # không làm gì, chứ không phải làm tất cả.
            if not (self.ham_xac_nhan(ten, ts) if self.ham_xac_nhan else False):
                return self._chan(ten, "xac_nhan", f"Người dùng từ chối {ten}")

        ham = self.so_dang_ky.get(ten)
        if ham is None:
            return {"ok": False, "loi": f"Không có công cụ {ten!r}"}
        try:
            kq = ham(**ts)
        except Exception as e:
            return {"ok": False, "loi": f"{type(e).__name__}: {e}"}

        if kq.get("ok") and ten in CONG_CU_DOI_THE_GIOI:
            self.da_thay_doi.append(ten)
        return kq


print("Bộ kiểm soát sẵn sàng.")

In [ ]:
# Chạy lại đúng agent ngốc lúc nãy, lần này có bộ kiểm soát
tg = TheGioi()
bks = BoKiemSoat(dang_ky(tg), ChinhSach(can_xac_nhan={"huy_don", "gui_email"}))
sdk_an_toan = {ten: (lambda t=ten: lambda **ts: bks.thuc_thi(t, ts))()
               for ten in dang_ky(tg)}

kq = chay_agent("Kho VX-DEN-03 hết rồi, đặt thêm cho đủ 50", model_ngoc, sdk_an_toan)
in_duong_di(kq)
print()
print("Tồn kho:", tg.kho["VX-DEN-03"]["ton"], "| Nhật ký chặn:", bks.nhat_ky_chan)

Cùng một model ngốc, nhưng giờ hành động bị chặn **trước khi công cụ chạy**,
và bạn có `nhat_ky_chan` để biết chuyện gì đã suýt xảy ra.

> 📌 `nhat_ky_chan` quan trọng ngang bản thân việc chặn. Nó là thứ trả lời câu
> hỏi *"tuần này agent đã cố làm gì mà bị chặn?"* — và câu trả lời đó thường chỉ
> ra chỗ prompt hoặc mô tả công cụ cần sửa.

## 4. Ba mức phân quyền

Không phải agent nào cũng cần mọi công cụ. Nguyên tắc **đặc quyền tối thiểu**
áp dụng nguyên vẹn ở đây.

In [ ]:
CHINH_SACH = {
    "chỉ đọc": ChinhSach(cong_cu_cho_phep={"tra_cuu_don", "tra_cuu_kho",
                                            "liet_ke_don"},
                          can_xac_nhan=set(), max_thay_doi=0),
    "bình thường": ChinhSach(cong_cu_cho_phep=None,
                              can_xac_nhan={"huy_don", "gui_email"},
                              max_thay_doi=3),
    "toàn quyền": ChinhSach(cong_cu_cho_phep=None, can_xac_nhan=set(),
                             max_thay_doi=99),
}

for ten_cs, cs in CHINH_SACH.items():
    tg = TheGioi()
    bks = BoKiemSoat(dang_ky(tg), cs, ham_xac_nhan=lambda t, ts: True)
    r = bks.thuc_thi("huy_don", {"ma_don": "DH1002"})
    print(f'{ten_cs:<14} huỷ đơn -> {"CHẠY" if r["ok"] else "CHẶN"}'
          f'   ({bks.nhat_ky_chan[0]["ly_do"] if bks.nhat_ky_chan else "-"})'
          f'   trạng thái đơn: {tg.don_hang["DH1002"]["trang_thai"]}')

> 📌 **Phần lớn agent trong thực tế nên bắt đầu ở mức "chỉ đọc".**
>
> Một agent tra cứu và báo cáo đã tạo ra rất nhiều giá trị mà gần như không có
> rủi ro. Chỉ mở quyền ghi khi bạn đã có bộ eval (bài 4) chứng minh nó hành xử
> đúng — chứ không phải khi nó "có vẻ thông minh".

## 5. Cạm bẫy kinh điển: hỏi mà thành làm

In [ ]:
tg = TheGioi()

def model_nhiet_tinh(lich_su):
    """Người dùng HỎI về hành động. Model LÀM luôn hành động đó."""
    if not any(b["vai"] == "cong_cu" for b in lich_su):
        return {"loai": "cong_cu", "ten": "huy_don", "tham_so": {"ma_don": "DH1004"}}
    return {"loai": "tra_loi", "noi_dung": "Tôi đã huỷ đơn DH1004 giúp bạn."}

# Không xác nhận
tg1 = TheGioi()
bks1 = BoKiemSoat(dang_ky(tg1), ChinhSach(can_xac_nhan=set()))
sdk1 = {t: (lambda tt=t: lambda **ts: bks1.thuc_thi(tt, ts))() for t in dang_ky(tg1)}
chay_agent("Nếu tôi muốn huỷ đơn DH1004 thì có được không?", model_nhiet_tinh, sdk1)
print("Không có cửa xác nhận -> DH1004:", tg1.don_hang["DH1004"]["trang_thai"])

# Có xác nhận, người nhìn thấy và từ chối
tg2 = TheGioi()
def nguoi_that(ten, ts):
    print(f'   [hỏi người] Cho phép {ten}({ts})? -> KHÔNG, tôi chỉ đang hỏi thôi')
    return False

bks2 = BoKiemSoat(dang_ky(tg2), ChinhSach(can_xac_nhan={"huy_don"}), nguoi_that)
sdk2 = {t: (lambda tt=t: lambda **ts: bks2.thuc_thi(tt, ts))() for t in dang_ky(tg2)}
chay_agent("Nếu tôi muốn huỷ đơn DH1004 thì có được không?", model_nhiet_tinh, sdk2)
print("Có cửa xác nhận       -> DH1004:", tg2.don_hang["DH1004"]["trang_thai"])

> ⚠️ **Đây là lỗi agent phổ biến nhất trong thực tế, và cũng tốn kém nhất.**
>
> Người dùng hỏi *"nếu tôi muốn huỷ thì có được không?"* — một câu hỏi. Model
> hiểu thành một mệnh lệnh và thực hiện. Với chatbot đây chỉ là hiểu nhầm; với
> agent đây là một đơn hàng bị huỷ thật.
>
> **Không có prompt nào chặn được việc này một cách chắc chắn.** Bạn có thể viết
> "chỉ hành động khi được yêu cầu rõ ràng" và nó sẽ đúng phần lớn thời gian —
> nhưng "phần lớn" không phải là một đảm bảo an toàn.
>
> Cửa xác nhận là thứ **chắc chắn**, vì nó không phụ thuộc vào việc model hiểu
> đúng hay sai.

Bài 4 sẽ đưa đúng tình huống này vào bộ eval, dưới tên ca `chi_hoi_ma_khong_lam`.

## 6. Xác nhận cho cái gì, và mệt mỏi vì xác nhận

Hỏi quá nhiều cũng nguy hiểm không kém hỏi quá ít: người dùng sẽ học cách bấm
"có" theo phản xạ, và lúc đó cửa xác nhận trở thành trang trí.

In [ ]:
print(f'{"công cụ":<22}{"đổi thế giới":>14}{"hoàn tác được":>16}{"nên hỏi?":>11}')
for ten in sorted(dang_ky(TheGioi())):
    doi = ten in CONG_CU_DOI_THE_GIOI
    hoan_tac = ten not in CONG_CU_KHONG_HOAN_TAC
    nen_hoi = "CÓ" if (doi and not hoan_tac) else ("cân nhắc" if doi else "không")
    print(f'{ten:<22}{"có" if doi else "không":>14}'
          f'{"được" if hoan_tac else "KHÔNG":>16}{nen_hoi:>11}')

**Quy tắc thực dụng: hỏi khi hành động vừa thay đổi thế giới, vừa không hoàn
tác được.**

- `cap_nhat_trang_thai` đổi dữ liệu nhưng **sửa lại được** → không cần hỏi mỗi lần
- `gui_email` **không thu hồi được** → luôn hỏi
- `huy_don` về mặt dữ liệu thì đảo được, nhưng về mặt nghiệp vụ thì khách đã nhận
  thông báo huỷ → hỏi

> 📌 Cột "hoàn tác được" nên là một thuộc tính bạn **khai báo cho từng công cụ**
> ngay khi viết nó, chứ không phải thứ đi suy luận lại sau này. Cùng với "có đổi
> thế giới không", hai cột đó quyết định gần như toàn bộ chính sách an toàn.

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **Không tin tham số của model** | Kiểm ở phía bạn, trước khi công cụ chạy |
| **Guardrail là một lớp riêng** | Không rải `if` trong từng công cụ — công cụ thứ tám sẽ thiếu |
| **Bốn cửa, đúng thứ tự** | Quyền → tham số → ngân sách → xác nhận |
| **Mặc định phải an toàn** | Không có hàm xác nhận = từ chối, không phải cho qua |
| **Đặc quyền tối thiểu** | Bắt đầu ở "chỉ đọc", mở quyền khi có eval chứng minh |
| **Hỏi mà thành làm** | Lỗi phổ biến nhất. Prompt không chặn chắc được — cửa xác nhận thì có |
| **Mệt mỏi vì xác nhận** | Hỏi quá nhiều thì người ta bấm "có" theo phản xạ |

**Bài tập:** `curriculum/09-agents/exercises/ex03_guardrails.py` (mục 3.1–3.3)

➡️ Bài tiếp: **[04 — Đánh giá agent](04_danh_gia_agent.ipynb)** ⭐⭐